# Problem Identification and Real-World Context

## Practical Setting

The retail beverage industry in Davao City, specifically the specialty coffee and matcha sector, has experienced exponential growth, leading to the growth of independent cafes and franchise kiosks across the city's commercial and residential corridors. For a business-to-business wholesale supplier such as a dairy distributor, coffee roaster, or packaging provider. Efficiently fulfilling daily or weekly restocking orders for hundreds of distinct retail locations presents a significant logistical challenge. Currently, many local suppliers rely on a single central warehouse, which leads to severe inefficiencies, high fuel consumption, 
and delayed deliveries due to urban traffic congestion.

To streamline the supply chain, the wholesale supplier plans to transition to a decentralized distribution model by opening 
$p$ micro-warehouses (or distribution hubs) strategically located across Davao City. 
The core decision-making problem is to determine exactly which candidate locations should be leased and how to assign each of the 400 existing coffee and matcha shops to these new hubs.

- The B2B Supplier/Logistics Provider: Seeking to minimize transportation costs and maximize operational efficiency.

- Retail Cafe Owners: Benefiting from faster, more reliable, and potentially cheaper supply deliveries, ensuring product freshness (especially for dairy and roasted beans).

- End Consumers: Ultimately benefiting from consistent product availability and stabilized retail prices.

# Case Study and Data Collection

## Project title: "Optimizing Supply Chain Networks for Coffee & Matcha Shops in Davao City"

The optimization model will be implemented using the DOCPLEX Python API to solve the $p$-median Facility Location Problem using empirical spatial data from the city.

## Data Sources and Relevance
To construct and evaluate the optimization model, two primary datasets were compiled:
- Demand Nodes (davao_coffee_matcha_nodes.csv):
    - Source & Content: Contains the geographical coordinates (Latitude/Longitude) of 400 existing coffee and matcha shops in Davao City, extracted from digital mapping platforms (e.g., Google Maps/OpenStreetMap).
    - Relevance: These represent the exact delivery destinations ($I$) that require regular restocking.
- Candidate Facilities (candidate_warehouses.csv):
    - Source & Content: Contains 52 selected candidate distribution hubs ($J$). These locations were mapped using commercial and industrial corridors, major highway intersections (e.g., Diversion Road, McArthur Highway, J.P. Laurel Ave), and key barangay centroids across Davao City's districts (from Toril to Bunawan).
    - These serve as the potential leasing sites for the micro-warehouses.

# Math Models

Problem is formulated as a classic $p$-median Facility Location Problem (FLP) using Integer Linear Programming (ILP). The $p$-median model seeks to locate a fixed number of $p$ distribution hubs among a set of candidate sites to minimize the total demand-weighted travel distance between all retail shops and their assigned supply hubs.

Sets and Indices

- $I$: Set of demand nodes (the 400 existing coffee and matcha shops in Davao City), indexed by $i \in \{0, 1, \dots, 399\}$.   
- $J$: Set of candidate facility locations (the 52 potential micro-warehouse sites across Davao City), indexed by $j \in \{0, 1, \dots, 51\}$.   

Parameters

- $w_i$: The demand weight at retail node $i \in I$, representing the estimated monthly inventory order volume (in kg/liters of coffee beans, matcha powder, and dairy) required by shop $i$.   
- $d_{ij}$: The spatial travel distance (in meters) between retail shop $i \in I$ and candidate warehouse $j \in J$, computed using the Haversine great-circle distance formula.   
- $p$: The exact number of micro-warehouses to be leased and operated across the city ($p \in \mathbb{Z}^+$, where $1 \leq p \leq \vert{}J\vert{}$).  

Decision VariablesThe model relies on two sets of binary decision variables to represent facility selection and supply chain routing:   

1. Facility Location Variable ($x_j$): Determines which candidate warehouses are leased.
   $$x_j = \begin{cases} 1, & \text{if a distribution hub is opened at candidate location } j \in J \\ 0, & \text{otherwise} \end{cases}$$
   
2. Node-Facility Assignment Variable ($y_{ij}$): Determines the logistical pairing between retail cafes and distribution hubs.
   $$y_{ij} = \begin{cases} 1, & \text{if coffee/matcha shop } i \in I \text{ is supplied by hub } j \in J \\ 0, & \text{otherwise} \end{cases}$$

## Objective Function$$\text{Minimize } Z = \sum_{j \in J} \sum_{i \in I} w_i d_{ij} y_{ij}$$

The objective function minimizes the total demand-weighted travel distance across the entire supply network. By multiplying the distance $d_{ij}$ by the shop's order volume $w_i$, the model prioritizes placing hubs closer to high-volume cafes and dense commercial clusters (such as Poblacion, Obrero, and Bajada). Minimizing $Z$ directly reduces total vehicle kilometer-tons, lowering fuel costs and delivery turnaround times.   

### Subject to Constraints:
- **Demand Assignment Constraint:**

$$\sum_{j \in J} y_{ij} = 1 \quad \forall i \in I$$

Every coffee and matcha shop $i$ must be assigned to and serviced by exactly one distribution hub. This prevents unfulfilled demand (no cafe is left without a supplier) and avoids split deliveries from multiple hubs, simplifying daily dispatch operations.  


- **Open Facility Coupling Constraint:**

$$y_{ij} \leq x_j \quad \forall i \in I, \; \forall j \in J$$

A retail shop $i$ can only be assigned to candidate location $j$ if a warehouse is actually opened at location $j$ ($x_j = 1$). If a candidate site is not leased ($x_j = 0$), then $y_{ij}$ is forced to $0$ for all shops, preventing ghost assignments to non-existent facilities.

- **($p$-facilities) Constraint:**

$$\sum_{j \in J} x_j = p$$

The supplier's capital expenditure budget restricts the network expansion to exactly $p$ active micro-warehouses. This allows management to evaluate network performance across fixed investment tiers (e.g., comparing a $p = 5$ hub network against a $p = 10$ hub network).



- **Integrality Constraints:**

$$x_j \in \{0, 1\} \quad \forall j \in J$$

$$y_{ij} \in \{0, 1\} \quad \forall i \in I, \; \forall j \in J$$


Enforces discrete combinatorial decision-making. A warehouse cannot be partially opened, and under the standard $p$-median framework, a cafe's supply contract is fulfilled wholly by its primary assigned hub.   

# Implementation

In [56]:
from docplex.mp.model import Model
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from haversine import haversine
import plotly.express as px
import plotly.graph_objects as go

## Create model

In [57]:
mdl = Model(name="coffee_matcha_flp")

In [58]:
nodes_df = pd.read_csv("davao_coffee_matcha_nodes_cut.csv", encoding="utf-8")
nodes_df.head()

,ID,place_name,latitude,longitude,demand_weight,LE_ENGLISH,X,Y,POPULATION
0,0,"10,000 Reasons Cafe",7.067546,125.613287,143,"10,000 Reasons Cafe",125.613287,7.067546,143
1,1,A Zen Cafe,7.086709,125.590856,143,A Zen Cafe,125.590856,7.086709,143
2,2,Annipie - SM Lanang,7.105201,125.602254,85,Annipie - SM Lanang,125.602254,7.105201,85
3,3,Antensity Brew,7.188219,125.451061,67,Antensity Brew,125.451061,7.188219,67
4,4,AtCurbside,7.081347,125.601935,121,AtCurbside,125.601935,7.081347,121


In [59]:
facilities_df = pd.read_csv("candidate_warehouses.csv", encoding="utf-8")
facilities_df.head()

,id,service_name,barangay,district,y,x,fixed_cost_php,capacity_kg,latitude,longitude
0,0,Sta. Ana Port Logistics Hub,Leon Garcia,Poblacion,7.0782,125.6235,65000,1800,7.0782,125.6235
1,1,Claveria Roastery & Supply Center,Barangay 32-D,Poblacion,7.0722,125.6124,61000,1500,7.0722,125.6124
2,2,Obrero Micro-Fulfillment Center,Barangay 18-B,Poblacion,7.0858,125.6145,59000,1500,7.0858,125.6145
3,3,Bajada Victoria Logistics Park,Barangay 20-B,Poblacion,7.0955,125.6132,64000,1750,7.0955,125.6132
4,4,Agdao Public Market Supply Hub,Agdao Proper,Agdao,7.0841,125.6248,54000,1550,7.0841,125.6248


## COnstants and Parameters

Extract the weight/demand $w_i$ at node $i$. Set $p$ (number of hubs to open) to 8.

In [60]:
nodes = nodes_df["ID"].tolist()
facilities = facilities_df["id"].tolist()
names = dict(zip(nodes_df["ID"], nodes_df["LE_ENGLISH"]))

nodes_df['POPULATION'] = nodes_df['POPULATION'].astype(str).str.replace(',', '', regex=False)
nodes_df['POPULATION'] = pd.to_numeric(nodes_df['POPULATION'], errors='coerce')
w = dict(zip(nodes_df["ID"], nodes_df["POPULATION"])) #1st iteration weights
#w = {i: 1 for i in nodes} ## Assumes every cafe orders the exact same volume (2nd iteration weights)

p = 8 #baselines instance starting 8 (1st iteration)
#p = 3 #iteration 3 budget scenario

Compute the distance matrix $d_{ij}$ between the demand nodes and the candidate facilities using `cdist` from `scipy.spatial.distance` with `metric=haversine`. Coordinates are passed in `(latitude, longitude)` decimal degrees (`Y`, `X`), and distances are converted from kilometers to integer meters.

In [61]:
node_coords = nodes_df.loc[:, ["Y", "X"]]
facility_coords = facilities_df.loc[:, ["y", "x"]]

# Haversine returns distance in kilometers; multiply by 1000 to convert to integer meters
d = cdist(node_coords, facility_coords, metric=haversine)
d = np.round(d * 1000, decimals=0).astype(int)

## Decision Variables
Binary variables $x_j$ for the candidate facilities.

In [62]:
x = mdl.binary_var_dict(facilities, lb=0, name="x_%s")

Binary variables $y_{ij}$ for the node-facility assignment.

In [63]:
y = mdl.binary_var_matrix(keys1=nodes, keys2=facilities, name="y_%s_%s")

## Constants

Node Assignment Constraints: $\sum_{j \in J} y_{ij} = 1 \quad \forall i \in I$

In [64]:
mdl.add_constraints(mdl.sum(y[i, j] for j in facilities) == 1 for i in nodes);

Open Facility Assignment Constraints: $y_{ij} \leq x_j \quad \forall i \in I, j \in J$

In [65]:
mdl.add_constraints(y[i, j] <= x[j]
    for i in nodes
    for j in facilities);

Number of Facilities Constraint: $\sum_{j \in J} x_j = p$

In [66]:
mdl.add_constraint(mdl.sum(x[j] for j in facilities) == p);

## Obj Function

Minimize total demand-weighted distance: $\sum_{j \in J}\sum_{i \in I} w_i d_{ij} y_{ij}$


In [67]:
mdl.minimize(mdl.sum(w[i] * d[i, j] * y[i, j] for i in nodes for j in facilities))

## Solve the model and Solution (1st Iteration) (p = 8, weights)

In [43]:
def retrieve_solution(mdl, facilities_df, nodes_df):
    solution_df = mdl.solution.as_df()
    y_df = solution_df.loc[solution_df['name'].str.contains("y")]
    x_df = solution_df.loc[solution_df['name'].str.contains("x")]
    x_df = x_df['name'].str.split('_', n=2, expand=True)
    x_df.drop(0, axis=1, inplace=True)

    # Retrieve the facilities that are selected
    inc_fac_idx = [int(x) for x in list(x_df[1])]
    inc_fac_coords = []
    for index in inc_fac_idx:
        inc_fac_coords.append((facilities_df.iloc[int(index)]['y'], facilities_df.loc[int(index)]['x']))
    inc_fac_df = pd.DataFrame(inc_fac_coords, columns=['lat', 'long'])
    inc_fac_df['id'] = [int(x) for x in inc_fac_idx]
    inc_fac_df = pd.merge(inc_fac_df['id'], facilities_df, on='id')

    # Retrieve the assigned demand nodes
    y_df = y_df[~(y_df['value'] <= 0.5)]  
    y_df = y_df['name'].str.split('_', n=3, expand=True)
    y_df.drop(0, axis=1, inplace=True)
    y_df = y_df.rename(columns={1: 'ID', 2: 'facility_id'}).reset_index()
    y_df.drop("index", axis=1, inplace=True)
    y_df['ID'] = y_df['ID'].astype('int64')
    y_df['facility_id'] = y_df['facility_id'].astype('int')
    y_df = pd.merge(y_df, nodes_df, on='ID')

    return inc_fac_df, y_df

In [15]:
def plot_assignments(inc_fac_df, y_df): 
    fig = px.scatter_map(
        inc_fac_df, lat="y", lon="x", hover_name="service_name",
        zoom=10.5, center=dict(lat=7.085, lon=125.58), height=850
    )
    fig.update_traces(marker_size=15, marker_color='rgb(0,0,255)')
    fig.update_layout(map_style="carto-positron", margin={"r": 0, "t": 0, "l": 0, "b": 0})

    for i in set(inc_fac_df["id"]):
        lats = []
        longs = []
        nodes_assigned = y_df.loc[y_df['facility_id'] == i]
        f_lat = inc_fac_df[inc_fac_df["id"] == i]['y'].iloc[0]
        f_long = inc_fac_df[inc_fac_df["id"] == i]['x'].iloc[0]
        for j in range(len(nodes_assigned)):
            n_lat = nodes_assigned.iloc[j]['Y']
            n_long = nodes_assigned.iloc[j]['X']
            lats.append(n_lat)
            longs.append(n_long)
            lats.append(f_lat)
            longs.append(f_long)
        
        fig.add_trace(go.Scattermap(
            mode="lines", lon=longs, lat=lats, 
            line={'width': 2, 'color': 'rgb(0,0,0)'}, showlegend=False
        ))

    fig.add_trace(go.Scattermap(
        mode="markers", lat=y_df['Y'].tolist(), lon=y_df['X'].tolist(),
        text=y_df['LE_ENGLISH'].tolist(),  
        marker={'size': 8, 'color': 'rgb(255,0,0)'}, showlegend=False
    ))

    fig.show()

In [16]:
print("kape-kape...matcha-matcha...")
mdl.parameters.timelimit = 3600  # 1-hour time limit per project guidelines
solution = mdl.solve(log_output=True)

if solution:
    print(f"Solution Status: {mdl.solve_details.status}")
    print(f"Computational Time: {mdl.solve_details.time:.4f} sec.")
    print(f"Objective Value (Total Weighted Distance in kg-meters): {mdl.objective_value:,.2f}")
    
    inc_fac_df, y_df = retrieve_solution(mdl, facilities_df, nodes_df)
    

    summary = y_df.groupby('facility_id').agg(
        cafes_assigned=('ID', 'count'),
        total_demand_kg=('POPULATION', 'sum')
    ).reset_index().merge(inc_fac_df[['id', 'service_name', 'barangay', 'district']], left_on='facility_id', right_on='id')
    display(summary[['facility_id', 'service_name', 'barangay', 'district', 'cafes_assigned', 'total_demand_kg']])
    
    plot_assignments(inc_fac_df, y_df)
else:
    print("No solution found.")

kape-kape...matcha-matcha...
Version identifier: 22.2.0.0 | 2026-07-01 | 7cae668b4
CPXPARAM_Read_DataCheck                          1
CPXPARAM_TimeLimit                               3600
Tried aggregator 1 time.
Reduced MIP has 961 rows, 915 columns, and 2715 nonzeros.
Reduced MIP has 915 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.00 sec. (3.40 ticks)
Found incumbent of value 4.6460563e+07 after 0.02 sec. (7.94 ticks)
Probing time = 0.00 sec. (0.84 ticks)
Tried aggregator 1 time.
Detecting symmetries...
Reduced MIP has 961 rows, 915 columns, and 2715 nonzeros.
Reduced MIP has 915 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.00 sec. (3.72 ticks)
Probing time = 0.00 sec. (0.84 ticks)
Clique table members: 960.
MIP emphasis: balance optimality and feasibility.
MIP search method: dynamic search.
Parallel mode: deterministic, using up to 16 threads.
Root relaxation solution time = 0.00 sec. (1.36 ticks)

        Nodes                             

,facility_id,service_name,barangay,district,cafes_assigned,total_demand_kg
0,1,Claveria Roastery & Supply Center,Barangay 32-D,Poblacion,14,1669
1,5,Damosa IT & Logistics Center,Angliongto,Buhangin,1,190
2,8,Buhangin Gym Commercial Hub,Buhangin Proper,Buhangin,6,731
3,9,Maa Diversion Freight Center,Ma-a,Talomo,6,652
4,11,Matina Town Square Micro-Hub,Matina Crossing,Talomo,13,1624
5,12,Bangkal McArthur Logistics,Talomo Proper,Talomo,11,1497
6,13,Toril Poblacion Logistics Center,Toril Proper,Toril,2,198
7,14,Mintal Commercial Supply Hub,Mintal,Tugbok,7,733


## Solve model and Solution (2nd Iteratiion) (p = 8, assumed all cafes are equal)

In [29]:
def retrieve_solution(mdl, facilities_df, nodes_df):
    solution_df = mdl.solution.as_df()
    y_df = solution_df.loc[solution_df['name'].str.contains("y")]
    x_df = solution_df.loc[solution_df['name'].str.contains("x")]
    x_df = x_df['name'].str.split('_', n=2, expand=True)
    x_df.drop(0, axis=1, inplace=True)

    # Retrieve the facilities that are selected
    inc_fac_idx = [int(x) for x in list(x_df[1])]
    inc_fac_coords = []
    for index in inc_fac_idx:
        inc_fac_coords.append((facilities_df.iloc[int(index)]['y'], facilities_df.loc[int(index)]['x']))
    inc_fac_df = pd.DataFrame(inc_fac_coords, columns=['lat', 'long'])
    inc_fac_df['id'] = [int(x) for x in inc_fac_idx]
    inc_fac_df = pd.merge(inc_fac_df['id'], facilities_df, on='id')

    # Retrieve the assigned demand nodes
    y_df = y_df[~(y_df['value'] <= 0.5)]  
    y_df = y_df['name'].str.split('_', n=3, expand=True)
    y_df.drop(0, axis=1, inplace=True)
    y_df = y_df.rename(columns={1: 'ID', 2: 'facility_id'}).reset_index()
    y_df.drop("index", axis=1, inplace=True)
    y_df['ID'] = y_df['ID'].astype('int64')
    y_df['facility_id'] = y_df['facility_id'].astype('int')
    y_df = pd.merge(y_df, nodes_df, on='ID')

    return inc_fac_df, y_df

In [30]:
def plot_assignments(inc_fac_df, y_df): 
    fig = px.scatter_map(
        inc_fac_df, lat="y", lon="x", hover_name="service_name",
        zoom=10.5, center=dict(lat=7.085, lon=125.58), height=850
    )
    fig.update_traces(marker_size=15, marker_color='rgb(0,0,255)')
    fig.update_layout(map_style="carto-positron", margin={"r": 0, "t": 0, "l": 0, "b": 0})

    for i in set(inc_fac_df["id"]):
        lats = []
        longs = []
        nodes_assigned = y_df.loc[y_df['facility_id'] == i]
        f_lat = inc_fac_df[inc_fac_df["id"] == i]['y'].iloc[0]
        f_long = inc_fac_df[inc_fac_df["id"] == i]['x'].iloc[0]
        for j in range(len(nodes_assigned)):
            n_lat = nodes_assigned.iloc[j]['Y']
            n_long = nodes_assigned.iloc[j]['X']
            lats.append(n_lat)
            longs.append(n_long)
            lats.append(f_lat)
            longs.append(f_long)
        
        fig.add_trace(go.Scattermap(
            mode="lines", lon=longs, lat=lats, 
            line={'width': 2, 'color': 'rgb(0,0,0)'}, showlegend=False
        ))

    fig.add_trace(go.Scattermap(
        mode="markers", lat=y_df['Y'].tolist(), lon=y_df['X'].tolist(),
        text=y_df['LE_ENGLISH'].tolist(),  
        marker={'size': 8, 'color': 'rgb(255,0,0)'}, showlegend=False
    ))

    fig.show()

In [31]:
print("kape-kape...matcha-matcha...")
mdl.parameters.timelimit = 3600  # 1-hour time limit per project guidelines
solution = mdl.solve(log_output=True)

if solution:
    print(f"Solution Status: {mdl.solve_details.status}")
    print(f"Computational Time: {mdl.solve_details.time:.4f} sec.")
    print(f"Objective Value (Total Weighted Distance in kg-meters): {mdl.objective_value:,.2f}")
    
    inc_fac_df, y_df = retrieve_solution(mdl, facilities_df, nodes_df)
    

    summary = y_df.groupby('facility_id').agg(
        cafes_assigned=('ID', 'count'),
        total_demand_kg=('POPULATION', 'sum')
    ).reset_index().merge(inc_fac_df[['id', 'service_name', 'barangay', 'district']], left_on='facility_id', right_on='id')
    display(summary[['facility_id', 'service_name', 'barangay', 'district', 'cafes_assigned', 'total_demand_kg']])
    
    plot_assignments(inc_fac_df, y_df)
else:
    print("No solution found.")

kape-kape...matcha-matcha...
Version identifier: 22.2.0.0 | 2026-07-01 | 7cae668b4
CPXPARAM_Read_DataCheck                          1
CPXPARAM_TimeLimit                               3600
Tried aggregator 1 time.
Reduced MIP has 961 rows, 915 columns, and 2715 nonzeros.
Reduced MIP has 915 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.00 sec. (3.40 ticks)
Found incumbent of value 385304.000000 after 0.00 sec. (7.94 ticks)
Probing time = 0.00 sec. (0.84 ticks)
Tried aggregator 1 time.
Detecting symmetries...
Reduced MIP has 961 rows, 915 columns, and 2715 nonzeros.
Reduced MIP has 915 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.02 sec. (3.76 ticks)
Probing time = 0.00 sec. (0.84 ticks)
Clique table members: 960.
MIP emphasis: balance optimality and feasibility.
MIP search method: dynamic search.
Parallel mode: deterministic, using up to 16 threads.
Root relaxation solution time = 0.00 sec. (1.38 ticks)

        Nodes                             

,facility_id,service_name,barangay,district,cafes_assigned,total_demand_kg
0,1,Claveria Roastery & Supply Center,Barangay 32-D,Poblacion,14,1669
1,7,Panacan Industrial Zone A,Panacan,Bunawan,1,48
2,8,Buhangin Gym Commercial Hub,Buhangin Proper,Buhangin,6,873
3,9,Maa Diversion Freight Center,Ma-a,Talomo,6,652
4,11,Matina Town Square Micro-Hub,Matina Crossing,Talomo,13,1624
5,12,Bangkal McArthur Logistics,Talomo Proper,Talomo,11,1497
6,13,Toril Poblacion Logistics Center,Toril Proper,Toril,2,198
7,14,Mintal Commercial Supply Hub,Mintal,Tugbok,7,733


## Solve model and SOlution (3rd Iteration) (p = 3, weights)

In [44]:
def retrieve_solution(mdl, facilities_df, nodes_df):
    solution_df = mdl.solution.as_df()
    y_df = solution_df.loc[solution_df['name'].str.contains("y")]
    x_df = solution_df.loc[solution_df['name'].str.contains("x")]
    x_df = x_df['name'].str.split('_', n=2, expand=True)
    x_df.drop(0, axis=1, inplace=True)

    # Retrieve the facilities that are selected
    inc_fac_idx = [int(x) for x in list(x_df[1])]
    inc_fac_coords = []
    for index in inc_fac_idx:
        inc_fac_coords.append((facilities_df.iloc[int(index)]['y'], facilities_df.loc[int(index)]['x']))
    inc_fac_df = pd.DataFrame(inc_fac_coords, columns=['lat', 'long'])
    inc_fac_df['id'] = [int(x) for x in inc_fac_idx]
    inc_fac_df = pd.merge(inc_fac_df['id'], facilities_df, on='id')

    # Retrieve the assigned demand nodes
    y_df = y_df[~(y_df['value'] <= 0.5)]  
    y_df = y_df['name'].str.split('_', n=3, expand=True)
    y_df.drop(0, axis=1, inplace=True)
    y_df = y_df.rename(columns={1: 'ID', 2: 'facility_id'}).reset_index()
    y_df.drop("index", axis=1, inplace=True)
    y_df['ID'] = y_df['ID'].astype('int64')
    y_df['facility_id'] = y_df['facility_id'].astype('int')
    y_df = pd.merge(y_df, nodes_df, on='ID')

    return inc_fac_df, y_df

In [45]:
def plot_assignments(inc_fac_df, y_df): 
    fig = px.scatter_map(
        inc_fac_df, lat="y", lon="x", hover_name="service_name",
        zoom=10.5, center=dict(lat=7.085, lon=125.58), height=850
    )
    fig.update_traces(marker_size=15, marker_color='rgb(0,0,255)')
    fig.update_layout(map_style="carto-positron", margin={"r": 0, "t": 0, "l": 0, "b": 0})

    for i in set(inc_fac_df["id"]):
        lats = []
        longs = []
        nodes_assigned = y_df.loc[y_df['facility_id'] == i]
        f_lat = inc_fac_df[inc_fac_df["id"] == i]['y'].iloc[0]
        f_long = inc_fac_df[inc_fac_df["id"] == i]['x'].iloc[0]
        for j in range(len(nodes_assigned)):
            n_lat = nodes_assigned.iloc[j]['Y']
            n_long = nodes_assigned.iloc[j]['X']
            lats.append(n_lat)
            longs.append(n_long)
            lats.append(f_lat)
            longs.append(f_long)
        
        fig.add_trace(go.Scattermap(
            mode="lines", lon=longs, lat=lats, 
            line={'width': 2, 'color': 'rgb(0,0,0)'}, showlegend=False
        ))

    fig.add_trace(go.Scattermap(
        mode="markers", lat=y_df['Y'].tolist(), lon=y_df['X'].tolist(),
        text=y_df['LE_ENGLISH'].tolist(),  
        marker={'size': 8, 'color': 'rgb(255,0,0)'}, showlegend=False
    ))

    fig.show()

In [46]:
print("kape-kape...matcha-matcha...")
mdl.parameters.timelimit = 3600  # 1-hour time limit per project guidelines
solution = mdl.solve(log_output=True)

if solution:
    print(f"Solution Status: {mdl.solve_details.status}")
    print(f"Computational Time: {mdl.solve_details.time:.4f} sec.")
    print(f"Objective Value (Total Weighted Distance in kg-meters): {mdl.objective_value:,.2f}")
    
    inc_fac_df, y_df = retrieve_solution(mdl, facilities_df, nodes_df)
    

    summary = y_df.groupby('facility_id').agg(
        cafes_assigned=('ID', 'count'),
        total_demand_kg=('POPULATION', 'sum')
    ).reset_index().merge(inc_fac_df[['id', 'service_name', 'barangay', 'district']], left_on='facility_id', right_on='id')
    display(summary[['facility_id', 'service_name', 'barangay', 'district', 'cafes_assigned', 'total_demand_kg']])
    
    plot_assignments(inc_fac_df, y_df)
else:
    print("No solution found.")

kape-kape...matcha-matcha...
Version identifier: 22.2.0.0 | 2026-07-01 | 7cae668b4
CPXPARAM_Read_DataCheck                          1
CPXPARAM_TimeLimit                               3600
Tried aggregator 1 time.
Reduced MIP has 961 rows, 915 columns, and 2715 nonzeros.
Reduced MIP has 915 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.00 sec. (3.40 ticks)
Found incumbent of value 3.9383070e+07 after 0.00 sec. (7.94 ticks)
Probing time = 0.00 sec. (0.84 ticks)
Tried aggregator 1 time.
Detecting symmetries...
Reduced MIP has 961 rows, 915 columns, and 2715 nonzeros.
Reduced MIP has 915 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.02 sec. (3.72 ticks)
Probing time = 0.00 sec. (0.84 ticks)
Clique table members: 960.
MIP emphasis: balance optimality and feasibility.
MIP search method: dynamic search.
Parallel mode: deterministic, using up to 16 threads.
Root relaxation solution time = 0.00 sec. (3.72 ticks)

        Nodes                             

,facility_id,service_name,barangay,district,cafes_assigned,total_demand_kg
0,1,Claveria Roastery & Supply Center,Barangay 32-D,Poblacion,35,4272
1,12,Bangkal McArthur Logistics,Talomo Proper,Talomo,16,2091
2,14,Mintal Commercial Supply Hub,Mintal,Tugbok,9,931


## Solve modle and SOlution (iteration 4) (p = 8, capacitated)

In [72]:
mdl_cap = Model(name="davao_p-median_capacitated")

x_cap = mdl_cap.binary_var_dict(facilities, lb=0, name="x_%s")
y_cap = mdl_cap.binary_var_matrix(keys1=nodes, keys2=facilities, name="y_%s_%s")

cap = dict(zip(facilities_df["id"], facilities_df["capacity_kg"]))

# Constraints
mdl_cap.add_constraints(mdl_cap.sum(y_cap[i, j] for j in facilities) == 1 for i in nodes)
mdl_cap.add_constraints(y_cap[i, j] <= x_cap[j] for i in nodes for j in facilities)
mdl_cap.add_constraint(mdl_cap.sum(x_cap[j] for j in facilities) == p)
mdl_cap.add_constraints(
    mdl_cap.sum(w[i] * y_cap[i, j] for i in nodes) <= cap[j] * x_cap[j]
    for j in facilities
)

mdl_cap.minimize(mdl_cap.sum(w[i] * d[i, j] * y_cap[i, j] for i in nodes for j in facilities))
sol_cap = mdl_cap.solve(log_output=True)

if sol_cap:
    print(f"Capacitated Objective Value: {mdl_cap.objective_value:,.2f} kg-meters")
    inc_fac_cap, y_df_cap = retrieve_solution(mdl_cap, facilities_df, nodes_df)
    plot_assignments(inc_fac_cap, y_df_cap)

    summary_cap = (
        y_df_cap.groupby('facility_id')
        .agg(
            cafes_assigned=('ID', 'count'),
            total_demand_kg=('POPULATION', 'sum')
        )
        .reset_index()
        .merge(
            inc_fac_cap[['id', 'service_name', 'barangay', 'district', 'capacity_kg']],
            left_on='facility_id',
            right_on='id'
        )
    )
    summary_cap['utilization_%'] = (summary_cap['total_demand_kg'] / summary_cap['capacity_kg'] *
    100).round(2)

    display(summary_cap[[
        'facility_id', 'service_name', 'barangay', 'district',
        'cafes_assigned', 'total_demand_kg', 'capacity_kg', 'utilization_%'
    ]])
else:
    print("No feasible solution found.")

Version identifier: 22.2.0.0 | 2026-07-01 | 7cae668b4
CPXPARAM_Read_DataCheck                          1
Tried aggregator 1 time.
Reduced MIP has 976 rows, 915 columns, and 3630 nonzeros.
Reduced MIP has 915 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.00 sec. (3.62 ticks)


Found incumbent of value 4.6460563e+07 after 0.02 sec. (8.67 ticks)
Probing time = 0.00 sec. (1.49 ticks)
Tried aggregator 1 time.
Detecting symmetries...
Reduced MIP has 976 rows, 915 columns, and 3630 nonzeros.
Reduced MIP has 915 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.02 sec. (4.11 ticks)
Probing time = 0.00 sec. (1.47 ticks)
Clique table members: 1036.
MIP emphasis: balance optimality and feasibility.
MIP search method: dynamic search.
Parallel mode: deterministic, using up to 16 threads.
Root relaxation solution time = 0.02 sec. (2.12 ticks)

        Nodes                                         Cuts/
   Node  Left     Objective  IInf  Best Integer    Best Bound    ItCnt     Gap

*     0+    0                       4.64606e+07        0.0000           100.00%
*     0+    0                       1.64427e+07        0.0000           100.00%
*     0+    0                       1.24310e+07        0.0000           100.00%
*     0+    0                       1.1

,facility_id,service_name,barangay,district,cafes_assigned,total_demand_kg,capacity_kg,utilization_%
0,1,Claveria Roastery & Supply Center,Barangay 32-D,Poblacion,12,1475,1500,98.33
1,5,Damosa IT & Logistics Center,Angliongto,Buhangin,1,190,2000,9.50
2,8,Buhangin Gym Commercial Hub,Buhangin Proper,Buhangin,6,731,1600,45.69
3,9,Maa Diversion Freight Center,Ma-a,Talomo,7,773,1850,41.78
4,11,Matina Town Square Micro-Hub,Matina Crossing,Talomo,13,1640,1650,99.39
5,12,Bangkal McArthur Logistics,Talomo Proper,Talomo,12,1554,1750,88.80
6,13,Toril Poblacion Logistics Center,Toril Proper,Toril,2,198,1900,10.42
7,14,Mintal Commercial Supply Hub,Mintal,Tugbok,7,733,1650,44.42


In [73]:
from docplex.mp.conflict_refiner import ConflictRefiner

conflicts = ConflictRefiner().refine_conflict(mdl_cap)
conflicts.display()

conflict(s): 0


# Analysis